# Konu 11 — Zincirleme: üretim hattı

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 8'de tek görsel, Konu 9'da bir ızgara, Konu 10'da bir video ürettik; her biri ayrı bir defterdi. Bugün bunları
tek bir **hatta** bağlıyoruz: bir brif dosyası → metin modeliyle sloganlar → görsel modeliyle her slogana bir görsel
→ (Bonus) videoya çevrilen bir görsel. Bir halkanın çıktısı, sıradaki halkanın girdisi.

Isınma, Adım 1 ve Adım 2 anahtarsız. Adım 3'ten sonrası `anahtar.txt` ister. Defterin tamamı 4 metin ve 3 görsel isteği
gönderir (Bonus 1 ile 4 görsel); Adım 6'da bilerek reddettirilen 3 istek de buna eklenir. Hepsi ücretsiz günlük kotanın küçük bir kısmı. Video (Bonus 2) varsayılan olarak kapalı.

## Isınma — Hat kâğıt üstünde

Önce sahte halkalarla: üç küçük fonksiyon, hiçbiri internete çıkmıyor. Bu hücre bir şey yazmaz, yalnızca
fonksiyonları tanımlar.

In [ ]:
def slogan_yaz(konu):
    return konu + ": burada zaman yavaşlar"

def yonerge_kur(slogan, stil):
    return stil + " poster, idea: " + slogan

def gorsel_ciz(yonerge):
    return "[görsel] " + yonerge

Şimdi zinciri kuruyoruz: her satır bir öncekinin sonucunu alıyor.

In [ ]:
konu = "sessiz bir çalışma kafesi"
slogan = slogan_yaz(konu)
yonerge = yonerge_kur(slogan, "risograph")
gorsel = gorsel_ciz(yonerge)

print(slogan)
print(yonerge)
print(gorsel)

**Tahmin et, sohbete yaz:** ilk satırdaki konuyu `"gece açık bir kitapçı"` yapıp yeniden çalıştırırsan üç satırdan kaçı değişir?
Dene.

Hattın başındaki tek girdi sonuna kadar taşınıyor. Bu yüzden hattın en önemli satırı brif: başta yanlış bir kelime,
sonda yanlış bir görsel demek.

Gerçek hatta her halka bir istek. Bugünkü hat 3 slogan ve 3 görsel istiyor. Defteri derste 5 kez baştan çalıştırdığını düşün.

**Tahmin et:** hiçbir şeyi saklamazsak kaç istek gider? Bitenleri saklarsak?

In [ ]:
slogan_istegi = 3
gorsel_istegi = 3
calistirma = 5

print("Hiçbir şeyi saklamadan:", calistirma * (slogan_istegi + gorsel_istegi))
print("Bitenleri saklayarak:", slogan_istegi + gorsel_istegi)

Görsel başına yaklaşık 58 neuron (Konu 9): saklamadan 15 görsel ~870 neuron, saklayarak 3 görsel ~174. Bir slogan bunun
yanında çok ucuz (10 neuron'dan az). Ama aşağıda göreceğiz: sloganı saklamanın asıl sebebi kota değil.

## Adım 1 — Brif dosyası: `ayarlar.json`

Bu klasörde `ayarlar.json` diye bir dosya var. VS Code'da aç: hattın bütün ayarları orada, koddan ayrı. Bir **yapılandırma
dosyası** (config): müşteri ya da konu değişince kodu değil bu dosyayı değiştiriyoruz.

Biçimi JSON (Konu 2 ve Konu 9): tek katman bir sözlük. Konu 9'da `json.load` ile günlüğü geri okumuştuk; aynısı:

In [ ]:
import json

with open("ayarlar.json", encoding="utf-8") as dosya:
    ayarlar = json.load(dosya)

print(ayarlar)

Dosyada `false` yazıyordu, ekranda `False` görünüyor: JSON'da `true` / `false` küçük harfle yazılır, Python'a `True` /
`False` olarak gelir.

Her ayarı bir değişkene alıyoruz; aşağıdaki hücreler bu adları kullanacak:

In [ ]:
calisma = ayarlar["calisma"]
konu = ayarlar["konu"]
stil = ayarlar["stil"]
slogan_sayisi = ayarlar["slogan_sayisi"]
sicaklik = ayarlar["sicaklik"]
tohum = ayarlar["tohum"]
video_istendi = ayarlar["video"]
hareket = ayarlar["hareket"]

print(konu, "/", stil)

`ayarlar.json`'u değiştirirsen bu iki hücreyi yeniden çalıştır; yoksa değişkenler eski değerleri taşır.

## Adım 2 — Çalışma klasörü

Hattın bütün çıktıları tek bir klasöre gidecek: `ciktilar/` içinde, adı `ayarlar.json`'daki `calisma` alanından gelen bir klasör.
`/` Konu 8'deki gibi yolları birleştirir. `parents=True`: `ciktilar/` henüz yoksa onu da kur.

In [ ]:
from pathlib import Path

klasor = Path("ciktilar") / calisma
klasor.mkdir(parents=True, exist_ok=True)
print(klasor)

Neden klasörün adına otomatik bir tarih eklemedik (Konu 8'deki gibi)? Çünkü her çalıştırmada yeni bir klasör açılırdı; içinde
hiçbir şey "zaten var" olmazdı ve hat her şeyi baştan üretirdi. Konu 9'un kuralı: **aynı iş, aynı ad**.

Tarihli bir arşiv istiyorsan tarihi adın içine sen yaz: `"calisma": "2026-11-04_kafe"`. Brifi değiştirdiğinde (konu, stil,
tohum) `calisma` adını da değiştir; yeni brif yeni klasörde başlar, eskisi olduğu gibi kalır.

## Adım 3 — Hazır halkalar

`halkalar.py` bu klasördeki yardımcı dosya. Eski konularda yazdığın üç fonksiyon içinde hazır:

- `modele_sor(soru, sicaklik)`: Konu 5'te yazdığın fonksiyon. Tek farkı: istek başarısız olursa çökmez, `None` verir.
- `sabirli_ciz(yonerge, tohum)`: Konu 9'da yazdığın fonksiyon; görseli en fazla üç kez dener.
- `video_uret(yonerge, yol, gorsel_yolu)`: Konu 10'da adım adım yaptığın işin (başlat, bekle, indir) tek fonksiyonluk hâli;
  videoyu üretip `yol`'a kaydeder (Bonus 2'de).

Anahtarı `../anahtar.txt`'den kendisi okur. Halkalar hazır; onları **birbirine bağlamak** bugün senin işin.

In [ ]:
from halkalar import modele_sor, sabirli_ciz, video_uret

Konu 5'teki slogan sorusunu bir kalıba çeviriyoruz: sabit kısım ve sonunda brifteki konu. Önce tek bir istek.

**Çalıştırmadan önce tahmin et:** model yalnızca sloganı mı yazacak, yoksa başına sonuna bir şey ekleyecek mi?

In [ ]:
soru = "Şu konu için tek bir kısa slogan yaz. Yalnızca sloganı yaz, açıklama ekleme: " + konu

cevap = modele_sor(soru, sicaklik)
print(cevap)

Model talimata her zaman uymaz: sloganı tırnak içine alabilir, başına "Slogan:" yazabilir, sonuna bir açıklama ekleyebilir.
Bu cevap bir sonraki halkaya (görsel yönergesine) gidecek; önce **temizliyoruz**. Konu 1'in araçları:
`.strip()` baştaki ve sondaki boşlukları, `.replace('"', "")` düz tırnakları siler.

In [ ]:
temiz = cevap.strip()
temiz = temiz.replace('"', "")
print(temiz)

Modelin cevabı Cloudflare'de varsayılan olarak 256 belirteçle (token) sınırlı. Slogana fazlasıyla yeter; ama modelden uzun bir
metin istersen cevap yarıda kesilir.

## Adım 4 — 1. halka: sloganlar (ara çıktıyı sakla)

Bir halkanın sonucu, hattın **ara çıktısı**. Sloganları bir dosyaya yazıp saklayacağız: `sloganlar.json`.
Fonksiyon her çağrıldığında önce sorar: **zaten var mı?**

- Dosya varsa: oku, ver; modele hiç istek gitmez (`return` fonksiyonu orada bitirir).
- Yoksa: `slogan_sayisi` kez sor (`range(slogan_sayisi)`: `slogan_sayisi` 3, yani "3 kez"), her cevabı temizleyip listeye
  ekle, listeyi dosyaya yaz, ver.
- Bir cevap gelmezse (`None`) o turu atla (`continue`); hat durmaz.

Dosyaya yazmak Konu 9'daki günlükle aynı: `json.dump(..., ensure_ascii=False, indent=2)`.

In [ ]:
slogan_dosyasi = klasor / "sloganlar.json"

def slogan_halkasi():
    if slogan_dosyasi.exists():
        with open(slogan_dosyasi, encoding="utf-8") as dosya:
            sloganlar = json.load(dosya)
        print("Dosyadan okundu, istek gitmedi")
        return sloganlar
    sloganlar = []
    for tur in range(slogan_sayisi):
        cevap = modele_sor(soru, sicaklik)
        if cevap is None:
            print("Slogan gelmedi, atlıyorum")
            continue
        slogan = cevap.strip()
        slogan = slogan.replace('"', "")
        sloganlar.append(slogan)
    with open(slogan_dosyasi, "w", encoding="utf-8") as dosya:
        json.dump(sloganlar, dosya, ensure_ascii=False, indent=2)
    print("Üretildi ve kaydedildi")
    return sloganlar

In [ ]:
sloganlar = slogan_halkasi()
print(sloganlar)

`ciktilar/kafe_afisi_1/sloganlar.json`'u VS Code'da aç: üç slogan orada.

**Tahmin et, sohbete yaz:** aynı çağrıyı bir kez daha yaparsak kaç istek gider? Sloganlar aynı mı kalır?

In [ ]:
sloganlar = slogan_halkasi()
print(sloganlar)

Yalnızca kota için değil, **tutarlılık** için de saklıyoruz. Sıcaklık 0.9; her istekte başka bir slogan gelir (Konu 5). Sloganları her
çalıştırmada yeniden üretseydik, `gorsel_1.jpg` dünkü sloganla, kayıt bugünkü sloganla kalırdı: görsel ile metin birbirini tutmazdı.

Kural: **pahalı ya da her seferinde farklı çıkan** ara çıktıyı sakla.

## Adım 5 — 2. halka: her slogana bir görsel

Önce her slogan için görsel yönergesini kuruyoruz: stil (İngilizce, ayarlardan) + slogan + "görselde yazı olmasın".
Bu bir kalıp (Konu 9'daki şablon): sabit kısım ve iki boşluk. Yönergeleri saklamıyoruz: sloganlardan ve ayarlardan her
seferinde aynı biçimde, bedavaya kurulur.

In [ ]:
yonergeler = []
for slogan in sloganlar:
    yonerge = f"{stil}, a poster illustration that expresses this idea: {slogan}, no text, no letters"
    yonergeler.append(yonerge)
    print(yonerge)

Sloganlar Türkçe, yönergenin geri kalanı İngilizce. Görsel model çoğunlukla İngilizce metinle eğitildi (Konu 7); Türkçe
sloganın **fikrini** ne kadar anlayacak, birazdan göreceğiz.

"Görselde yazı olmasın" bilerek orada: tipografiyi modele bırakmıyoruz. Görsel modeller harfleri çoğu zaman bozar; sloganı
afişe tasarımcı dizer. Hat **taslak** üretir.

Görsel halkası tek bir görsel için çalışan bir fonksiyon. Slogan halkasıyla aynı soruyu sorar: zaten var mı? Varsa atla
(Konu 9'daki `yol.exists()`), yoksa üret ve kaydet. Her seferinde bir **kayıt** (sözlük) verir: hangi halka, ne girdi,
hangi dosya çıktı, sonuç ne (`durum`).

- `str(yol)`: dosya yolunu metne çevirir. Kayıt sonunda JSON'a yazılacak; JSON `Path`'i tanımaz, metni tanır.
- `return kayit` fonksiyonu orada bitirir; hangi yoldan çıkarsa çıksın elimize bir kayıt gelir.

In [ ]:
def gorsel_halkasi(sira, yonerge):
    yol = klasor / f"gorsel_{sira}.jpg"
    kayit = {"halka": "gorsel", "girdi": yonerge, "cikti": str(yol), "durum": "denenmedi"}
    if yol.exists():
        kayit["durum"] = "uretildi"
        print("Zaten var, atlıyorum:", yol)
        return kayit
    resim = sabirli_ciz(yonerge, tohum)
    if resim is None:
        kayit["durum"] = "olmadi"
        print("Olmadı, atlıyorum:", yol)
        return kayit
    with open(yol, "wb") as dosya:
        dosya.write(resim)
    kayit["durum"] = "uretildi"
    print("Kaydedildi:", yol)
    return kayit

Şimdi zinciri kuruyoruz: her yönerge görsel halkasına gidiyor, gelen kayıt listeye ekleniyor. `sira` her turun **başında**
bir artıyor: ilk görsel `gorsel_1.jpg`. Üç görsel bir dakika kadar sürer.

In [ ]:
kayitlar = []
sira = 0
for yonerge in yonergeler:
    sira = sira + 1
    kayit = gorsel_halkasi(sira, yonerge)
    kayitlar.append(kayit)

print(len(kayitlar), "kayıt")

`ciktilar/kafe_afisi_1/` klasöründeki görsellere bak. Sohbete yaz:

- Görselde yine de yazı çıktı mı? Harfler okunuyor mu?
- Görseller sloganların **fikrini** mi taşıyor, yoksa yalnızca stili mi? Üç görsel birbirinden ne kadar farklı?

## Adım 6 — Bir halka koparsa hat durmasın

Gerçek bir hatta bir halka er geç kopar: internet gider, kota biter, bir yönerge reddedilir. Derste bunu beklemek yerine bilerek
koparıyoruz: dördüncü görseli **boş bir yönergeyle** istiyoruz. Görsel model boş yönergeyi kabul etmez.

**Tahmin et:** hücre hata verip duracak mı? Kayıtta ne yazacak? (Üç denemeyle 10 saniye kadar sürer.)

In [ ]:
kayit = gorsel_halkasi(4, "")
kayitlar.append(kayit)

print(kayit)
print(len(kayitlar), "kayıt")

Sunucu üç kez de reddetti (durum kodu 400). Ama hücre çökmedi: `gorsel_halkasi` hata yerine `"durum": "olmadi"` yazan bir
kayıt verdi. Bir döngünün içinde olsaydık döngü sıradaki görsele geçerdi. Hat iki şeyi birlikte yapıyor:

1. Kopuk halkayı **kayda geçiriyor** (hangi görsel, `durum`: `olmadi`).
2. Kopmayanlarla **devam ediyor**.

`gorsel_4.jpg` hiç oluşmadı; hattı yarın yeniden çalıştırırsan yalnızca o denenir.

## Adım 7 — Kayıt: `uretim.json`

Bütün hattın kaydı tek dosyada: hangi ayarlarla çalıştı, hangi sloganlar çıktı, her görselin sonucu ne. Konu 9'daki günlüğün
aynısı; bu kez bir sözlüğün içinde üç bölüm.

In [ ]:
gunluk = {"ayarlar": ayarlar, "sloganlar": sloganlar, "kayitlar": kayitlar}

with open(klasor / "uretim.json", "w", encoding="utf-8") as dosya:
    json.dump(gunluk, dosya, ensure_ascii=False, indent=2)

print("Kayıt yazıldı:", len(kayitlar), "kayıt")

`ciktilar/kafe_afisi_1/uretim.json`'u VS Code'da aç. Program da onu geri okuyup eksikleri bulabilir:

In [ ]:
with open(klasor / "uretim.json", encoding="utf-8") as dosya:
    okunan = json.load(dosya)

okunan_kayitlar = okunan["kayitlar"]
for kayit in okunan_kayitlar:
    if kayit["durum"] == "olmadi":
        print("Eksik:", kayit["cikti"])

Final projesinin teslimindeki **üretim günlüğü** bu dosya: hangi brif, hangi model ayarı, ne çıktı, ne eksik kaldı.

## Adım 8 — Pano: slogan ve görsel yan yana

Konu 9'da yazdığın kontak sayfası, bu kez tek satır: her kutuda bir görsel, üstünde onu doğuran slogan.
`plt.imread` görseli okur, `plt.imshow` kutuya koyar; dosya yoksa kutu boş kalır.

In [ ]:
import matplotlib.pyplot as plt

sutun = len(sloganlar)

plt.figure(figsize=(12, 5))
sira = 0
for slogan in sloganlar:
    sira = sira + 1
    yol = klasor / f"gorsel_{sira}.jpg"
    plt.subplot(1, sutun, sira)
    if yol.exists():
        resim = plt.imread(yol)
        plt.imshow(resim)
    plt.title(slogan, fontsize=9)
    plt.axis("off")

plt.savefig(klasor / "pano.png", bbox_inches="tight")
plt.show()

Panoya tasarımcı gözüyle bak. Hangi slogan hangi görselle güçlü? Slogan ile görselin **aynı şeyi söylediği** kutu mu daha iyi,
yoksa birbirini **tamamladığı** kutu mu (görsel bir şey gösteriyor, slogan ona yeni bir anlam ekliyor)? Sohbete tek cümle.

Videoya çevirmek isteyeceğin görseli aşağıya yaz (Bonus 2 bunu kullanır):

In [ ]:
secilen = klasor / "gorsel_1.jpg"

## Bonus

**1. Ara çıktıya elle dokun.** Ara çıktıyı saklamanın bir faydası daha var: hattın ortasında araya girebilirsin.

1. `ciktilar/kafe_afisi_1/sloganlar.json`'u VS Code'da aç, ikinci sloganı kendi yazdığın bir sloganla değiştir, kaydet.
   (Tırnakları ve virgülleri bırak; JSON biçimi bozulmasın.)
2. Aynı klasördeki `gorsel_2.jpg`'yi sil.
3. Adım 4'ün ilk çağrı hücresinden (`sloganlar = slogan_halkasi()`) Adım 5'in sonuna kadar sırayla yeniden çalıştır,
   sonra Adım 8'in pano hücresini.

Kaç istek gitti? Pano neden yalnızca bir kutuda değişti?

**2. 3. halka: video.** Varsayılan olarak kapalı: video hocanın hesabından, ücretli üretiliyor ve sayısı sınırlı (Konu 10).
Açmak için `ayarlar.json`'da `"video": false` satırını `"video": true` yap, kaydet, Adım 1'in iki hücresini yeniden çalıştır.
Video halkası aynı kalıpla yazılıyor; "zaten var mı?" sorusunu Konu 10'daki `video_uret` kendisi soruyor (video pahalı).
Halkanın girdisi, görsel halkasının çıktısı (`secilen`); her yoldan bir kayıt çıkıyor.

In [ ]:
def video_halkasi(gorsel_yolu):
    video_yolu = klasor / "video.mp4"
    kayit = {"halka": "video", "girdi": str(gorsel_yolu), "cikti": str(video_yolu), "durum": "denenmedi"}
    sonuc = video_uret(hareket, video_yolu, gorsel_yolu)
    if sonuc is None:
        kayit["durum"] = "olmadi"
    else:
        kayit["durum"] = "uretildi"
    return kayit

Video kapalıysa yalnızca bir uyarı yazar. Açıksa birkaç dakika sürebilir (Konu 10'daki bekleme).

In [ ]:
if video_istendi:
    kayit = video_halkasi(secilen)
    kayitlar.append(kayit)
else:
    print("Video kapalı: ayarlar.json'da video false")

Kaydı videoyla birlikte yeniden yaz:

In [ ]:
gunluk = {"ayarlar": ayarlar, "sloganlar": sloganlar, "kayitlar": kayitlar}

with open(klasor / "uretim.json", "w", encoding="utf-8") as dosya:
    json.dump(gunluk, dosya, ensure_ascii=False, indent=2)

print("Kayıt güncellendi:", len(kayitlar), "kayıt")

## Sıradaki konu

Bugün hangi halkanın ne zaman çalışacağına **sen** karar verdin: önce slogan, sonra görsel, sonra video. Konu 12'de bu kararı
modele bırakıyoruz: modele "elinde şu araçlar var" deyip kullanıcının cümlesine göre hangisini çağıracağını ona seçtireceğiz.